In [1]:
import uuid
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Import hàm build graph thay vì import graph trực tiếp
# from agents.hotel.agent import hotel_graph as graph 
from agents.travel_planner.agent import build_travel_planner_graph 
from utils.tracing import with_trace_config

# Khởi tạo graph (cần dùng await vì hàm này là async)
graph = await build_travel_planner_graph()

In [2]:
from langchain_core.messages import AIMessage


def _content_to_text(content) -> str:
    """Chuyển content (str hoặc list block) thành chuỗi thuần để print xuống dòng đúng."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict):
                parts.append(block.get("text", ""))
            else:
                parts.append(str(block))
        return "".join(parts)
    return str(content)


async def stream_graph_updates(inputs: dict):
    thread_id = str(uuid.uuid4())
    config = with_trace_config(
        {"configurable": {"thread_id": thread_id}},
        run_name="travel_planner_notebook",
        tags=["customer-support", "travel-planner", "notebook"],
        metadata={"thread_id": thread_id, "agent": "travel_planner"},
    )

    async for event in graph.astream(inputs, config=config):
        if "travel_planner_chat" in event:
            for msg in event["travel_planner_chat"].get("messages", []):
                if isinstance(msg, AIMessage) and msg.tool_calls:
                    for tc in msg.tool_calls:
                        print("Tool:", tc["name"])
                        print("Args:", tc["args"])

                if hasattr(msg, "content") and msg.content:
                    print("Assistant:", _content_to_text(msg.content))

        if "tools" in event:
            for msg in event["tools"].get("messages", []):
                if hasattr(msg, "content") and msg.content:
                    text = _content_to_text(msg.content)
                    preview = text if len(text) < 500 else text[:500] + "..."
                    print("Tool Output:", preview)

In [ ]:
test_cases = [

    "Tôi muốn lên plan cho 2 ngày ở Hà Nội, xuất phát từ Sài Gòn, từ 25/7/2026 đến 27/7/2026."
]


for q in test_cases:
    print(f"\n🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    print(inputs)
    await stream_graph_updates(inputs) 


🧑‍💻 User: Tôi muốn lên plan cho 2 ngày ở Hà Nội, xuất phát từ Sài Gòn, từ 25/7/2026 đến 27/7/2026.
{'messages': [{'role': 'user', 'content': 'Tôi muốn lên plan cho 2 ngày ở Hà Nội, xuất phát từ Sài Gòn, từ 25/7/2026 đến 27/7/2026.'}]}
Tool: get_weather_tool
Args: {'days': 3, 'location': 'Ha Noi'}
Tool Output: {
  "location": {
    "name": "Ha Noi",
    "region": "",
    "country": "Vietnam",
    "localtime": "2026-07-17 15:53"
  },
  "current": {
    "temperature_c": 33.3,
    "feels_like_c": 41.4,
    "condition": "Có mây rải rác",
    "icon": "https://cdn.weatherapi.com/weather/64x64/day/116.png",
    "humidity": 63,
    "wind_kph": 13.3,
    "precipitation_mm": 0.0,
    "uv": 4.5
  },
  "forecast": [
    {
      "date": "2026-07-17",
      "min_temp_c": 27.6,
      "max_temp_c": 32.7,
      "condi...
Tool: search_attractions_tool
Args: {'location': 'Ha Noi'}
Tool Output: {
  "source": "booking_com15_rapidapi",
  "product_id": "PRIY47gaM78X",
  "external_attraction_id": "PRIY47gaM78

: 